# End-To-End Capture, Visualization, And Prediction Demo

Purpose: provide one notebook that shows the full local workflow: capture planning, loading a recording, signal visualisation, and prediction summaries for person/motion/breathing.

Run path: from the repo root, start Jupyter with `uv run jupyter lab`. The notebook analyzes an existing local CSV by default; use the printed capture command to create a fresh recording outside the notebook.

Fixture / simulated source: auto-discovers local ESP32 CSI CSV files under `data/recordings/`. The current demo fixture is `data/recordings/esp32_csi/dummy-test/rx01_desk_csi.csv` when present.

Expected interpretation: the first half checks capture quality and visualizes raw CSI structure; the second half runs current heuristic predictions. Treat breathing results as exploratory unless packet timing and link identity are clean.

Limitations: this is a demo workflow, not a validated biometric pipeline. Pose estimation and three-receiver position triangulation remain separate notebooks because they need trained pose data or calibrated receiver geometry.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

try:
    from ruview.hardware.esp32_capture_analysis import analyze_capture, load_esp32_capture
    from ruview.vitals.breathing import extract_breathing_residual
    from ruview.vitals.preprocessing import CsiVitalPreprocessor
    from ruview.vitals.quality import estimate_rate_from_samples
except Exception as exc:
    raise RuntimeError('Run this notebook from the repo environment, e.g. `uv run jupyter lab`.') from exc

In [ ]:
def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'ruview').exists():
            return candidate
    return current


repo_root = find_repo_root()

# Recording source. Leave RECORDING_CSV as None to auto-pick a local CSV.
USE_RECORDING = True
RECORDING_CSV = None

# Live capture planning values. The command below is printed, not executed.
SERIAL_PORT = '/dev/cu.usbserial-0001'
RECEIVER_ID = 'rx01_desk'
SESSION_ID = 'demo-test'
MAX_ROWS = 1200

capture_command = (
    f'uv run --extra capture scripts/collect_esp32_csi.py {RECEIVER_ID} '
    f'--port {SERIAL_PORT} --session-id {SESSION_ID} --max-rows {MAX_ROWS} '
    f'--label person_or_empty --notes "demo capture"'
)

print('To collect a new recording, run this in a terminal:')
print('  ' + capture_command)

candidates = sorted((repo_root / 'data' / 'recordings').glob('**/*_csi.csv'))
if not USE_RECORDING:
    raise ValueError('This demo notebook is recording-backed; keep USE_RECORDING=True or use notebooks 01-06 for synthetic fixtures.')

recording_path = Path(RECORDING_CSV).expanduser().resolve() if RECORDING_CSV else None
if recording_path is None and candidates:
    recording_path = candidates[0]
if recording_path is None:
    raise FileNotFoundError('No local *_csi.csv recording found under data/recordings/.')

print('\nAnalyzing local recording:')
print('  ' + str(recording_path))

In [ ]:
capture = load_esp32_capture(recording_path)
summary, predictions = analyze_capture(capture, window_size=64, step_size=16)

summary_dict = json.loads(summary.to_json())
print(json.dumps(summary_dict, indent=2))

print('\nQuick prediction summary:')
print(f"  predicted_state={summary.predicted_state}")
print(f"  presence_score_mean={summary.presence_score_mean:.3f}")
print(f"  presence_score_max={summary.presence_score_max:.3f}")
print(f"  motion_score_mean={summary.motion_score_mean:.3f}")
print(f"  motion_score_max={summary.motion_score_max:.3f}")

In [ ]:
def elapsed_seconds(capture):
    real_ts = capture.timestamps
    if np.isfinite(real_ts).sum() >= 2 and float(np.nanmax(real_ts) - np.nanmin(real_ts)) > 0:
        return real_ts - float(np.nanmin(real_ts))
    mono = capture.host_monotonic_ns.astype(np.float64)
    return (mono - float(mono[0])) / 1_000_000_000.0


elapsed = elapsed_seconds(capture)
amp = np.where(capture.valid_mask, capture.amplitude, np.nan)
phase = np.where(capture.valid_mask, capture.phase, np.nan)
max_cols = min(amp.shape[1], 128)
mid = [(item.start_seconds + item.end_seconds) / 2.0 for item in predictions]

fig, axes = plt.subplots(4, 1, figsize=(12, 13), constrained_layout=True)
axes[0].plot(elapsed, capture.rssi, linewidth=1)
axes[0].set_title('RSSI over time')
axes[0].set_xlabel('seconds')
axes[0].set_ylabel('dBm')

im1 = axes[1].imshow(amp[:, :max_cols].T, aspect='auto', origin='lower', interpolation='nearest')
axes[1].set_title('CSI amplitude heatmap')
axes[1].set_xlabel('packet index')
axes[1].set_ylabel('subcarrier index')
fig.colorbar(im1, ax=axes[1], label='amplitude')

im2 = axes[2].imshow(phase[:, :max_cols].T, aspect='auto', origin='lower', interpolation='nearest', cmap='twilight')
axes[2].set_title('CSI phase heatmap')
axes[2].set_xlabel('packet index')
axes[2].set_ylabel('subcarrier index')
fig.colorbar(im2, ax=axes[2], label='radians')

axes[3].plot(mid, [item.presence_score for item in predictions], label='presence')
axes[3].plot(mid, [item.motion_score for item in predictions], label='motion')
axes[3].axhline(0.35, color='black', linestyle='--', linewidth=1, label='presence threshold')
axes[3].set_title('Windowed prediction scores')
axes[3].set_xlabel('seconds')
axes[3].set_ylabel('score')
axes[3].legend();

In [ ]:
def filled_amplitude(capture, indices):
    values = capture.amplitude[indices].astype(np.float64, copy=True)
    mask = capture.valid_mask[indices]
    values[~mask] = np.nan
    valid = np.isfinite(values)
    counts = valid.sum(axis=0)
    sums = np.where(valid, values, 0.0).sum(axis=0)
    col_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    rows, cols = np.where(~valid)
    values[rows, cols] = col_means[cols]
    return values


def scalar_residual_stream(capture, indices):
    values = filled_amplitude(capture, indices)
    preprocessor = CsiVitalPreprocessor(n_subcarriers=values.shape[1], alpha=0.05)
    samples = []
    for row in values:
        residuals = preprocessor.process(row)
        samples.append(float(extract_breathing_residual(residuals if residuals is not None else [])))
    return np.asarray(samples, dtype=np.float64)


def resample_uniform(times, values):
    order = np.argsort(times)
    times = np.asarray(times, dtype=np.float64)[order]
    values = np.asarray(values, dtype=np.float64)[order]
    keep = np.isfinite(times) & np.isfinite(values)
    times, values = times[keep], values[keep]
    unique = np.concatenate([[True], np.diff(times) > 1e-9]) if times.size else np.array([], dtype=bool)
    times, values = times[unique], values[unique]
    if times.size < 4:
        return times, values, 0.0
    duration = float(times[-1] - times[0])
    rate = (times.size - 1) / duration if duration > 0.0 else 0.0
    if rate <= 0.0:
        return times, values, 0.0
    grid = np.linspace(times[0], times[-1], max(int(round(duration * rate)) + 1, times.size))
    return grid, np.interp(grid, times, values), rate


link_keys = [(row.get('mac', ''), row.get('channel', '')) for row in capture.rows]
dominant_key, dominant_count = Counter(link_keys).most_common(1)[0]
dominant_indices = np.asarray([idx for idx, key in enumerate(link_keys) if key == dominant_key], dtype=int)
all_indices = np.arange(len(capture.rows))

print(f'Dominant link: mac={dominant_key[0]}, channel={dominant_key[1]}, packets={dominant_count}')

In [ ]:
breathing_results = []
for name, indices in [('dominant_link', dominant_indices), ('all_packets_mixed_links', all_indices)]:
    times = elapsed[indices]
    residual = scalar_residual_stream(capture, indices)
    grid, signal, sample_rate = resample_uniform(times, residual)
    if signal.size < 4 or sample_rate <= 0.0:
        estimate = None
    else:
        high = min(0.5, 0.45 * sample_rate)
        estimate = None if high <= 0.1 else estimate_rate_from_samples(
            signal,
            sample_rate_hz=sample_rate,
            band_hz=(0.1, high),
            min_duration_seconds=10.0,
            min_samples=max(int(round(10.0 * sample_rate)), 4),
        )
    breathing_results.append((name, grid, signal, sample_rate, estimate))

for name, _grid, _signal, sample_rate, estimate in breathing_results:
    print('\n' + name)
    print(f'  sample_rate={sample_rate:.2f} Hz')
    if estimate is None or not estimate.is_available:
        print('  breathing: unavailable')
    else:
        print(f'  breathing: {estimate.value_bpm:.1f} BPM, confidence={estimate.confidence:.2f}, status={estimate.status.value}')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
for name, grid, signal, sample_rate, estimate in breathing_results:
    axes[0].plot(grid, signal, label=name)
    if signal.size >= 4 and sample_rate > 0.0:
        centered = signal - float(np.mean(signal))
        n_fft = 1 << (max(centered.size, 4) - 1).bit_length()
        padded = np.zeros(n_fft, dtype=np.float64)
        padded[: centered.size] = centered * np.hanning(centered.size)
        spectrum = np.fft.rfft(padded)
        freqs_bpm = np.fft.rfftfreq(n_fft, d=1.0 / sample_rate) * 60.0
        power = spectrum.real * spectrum.real + spectrum.imag * spectrum.imag
        mask = (freqs_bpm >= 6.0) & (freqs_bpm <= 30.0)
        axes[1].plot(freqs_bpm[mask], power[mask], label=name)
axes[0].set_title('Breathing residual streams')
axes[0].set_xlabel('seconds')
axes[0].set_ylabel('residual amplitude')
axes[0].legend()
axes[1].set_title('Breathing-band spectrum')
axes[1].set_xlabel('breaths per minute')
axes[1].set_ylabel('power')
axes[1].legend();